# AGRIVISION Model 2 - Plant Health Classification

Trains the plant-health classifier that runs **after** Model 1 (COCO-SSD) has
confirmed a plant is in frame, and produces the two files the browser app
loads:

```
model/plant_health_classifier.tflite
model/class_names.json
```

**Runtime -> Change runtime type -> GPU** before running anything. On a T4 the
whole notebook takes roughly 40-70 minutes, most of it the one-time PlantVillage
download and the fine-tuning phase.

Run the cells in order. Cell 9 prints the complete report - dataset, accuracy,
precision/recall/F1, confusion matrices and the generalisation verdict. Nothing
in this notebook invents a number: every figure printed comes from an actual
evaluation run.

### Read this before trusting the accuracy

PlantVillage is **laboratory imagery** - one detached leaf on a uniform
background. Noyan (2022, arXiv:2206.04374) trained a classifier on *8 background
pixels alone* and got 49% where random is 2.6%, i.e. the backgrounds leak the
label. Models scoring ~99% in-domain have measured ~31% on other datasets.

So the PlantVillage test score is **not** evidence the model works on your
webcam. The PlantDoc field-photo score is the honest number, and this notebook
reports it as the headline.


## 1. Get the training code

In [ ]:
# Clone the repo that holds training/plant_health/.
#
# IMPORTANT: Colab clones from GitHub, not from your laptop - commit and push
# your local changes FIRST or you will train with stale scripts:
#     git add -A && git commit -m "model 2 pipeline" && git push
REPO_URL = "https://github.com/navyashreebh1-lang/cv.git"
BRANCH   = "main"

import os, sys, shutil, subprocess
from pathlib import Path

if REPO_URL:
    if not Path("cv-prototype").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, "cv-prototype"], check=True)
    PH = Path("cv-prototype/training/plant_health").resolve()
else:
    # Fallback: zip training/plant_health on your machine and upload it here.
    from google.colab import files
    if not Path("plant_health").exists():
        print("Upload a zip of training/plant_health ...")
        up = files.upload()
        shutil.unpack_archive(next(iter(up)), ".")
    PH = Path("plant_health").resolve()

assert (PH / "config.py").exists(), f"config.py not found under {PH}"
os.chdir(PH)
sys.path.insert(0, str(PH))
print("Working dir:", Path.cwd())
print("Scripts    :", sorted(p.name for p in (PH / "scripts").glob("*.py")))

## 2. Environment check

Stops here if the runtime has no GPU - CPU training on 54k images is hours, not minutes.

In [ ]:
!pip install -q tensorflow_datasets scikit-learn matplotlib pillow

import tensorflow as tf, keras, platform
print("Python     :", platform.python_version())
print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU        :", gpus or "NONE")
if not gpus:
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> Hardware accelerator: GPU, "
        "then Runtime -> Restart session and run this cell again.")

## 3. Prepare the dataset

Fetches PlantVillage with a sparse `git clone` of the authors' GitHub repo
(`spMohanty/PlantVillage-Dataset`, `raw/color/` only) - no Kaggle account, API
token or login - and clones PlantDoc for out-of-domain evaluation, then cleans
and splits:

> TFDS is no longer the primary source: it pulls the archive from
> `data.mendeley.com`, which now returns **HTTP 403**. It stays available as a
> fallback via `--pv-source tfds`.

* corrupt / tiny / near-blank images dropped
* near-duplicates found with a 64-bit difference hash
* **splits assigned per duplicate cluster, never per image** - PlantVillage has
  many near-identical shots of the same physical leaf, and splitting per image
  puts the same leaf in train and test, which is a large part of why published
  PlantVillage accuracies look so good
* classes under 150 usable images are dropped rather than trained badly

In [ ]:
!python scripts/prepare_dataset.py

### 3b. What actually got prepared

Reads the manifest the previous cell wrote - class balance, split sizes, and whether the out-of-domain set exists at all.

In [ ]:
import json
from pathlib import Path
import config as C

man = json.loads((C.DATASET_DIR / "manifest.json").read_text())
rep = man["prepare_report"]

print(f"classes kept      : {man['num_classes']}  (of {rep['classes_seen']} seen)")
print(f"crops             : {', '.join(man['crops'])}")
print(f"healthy images    : {man['healthy_images']:,}")
print(f"unhealthy images  : {man['unhealthy_images']:,}")
print(f"OOD (PlantDoc)    : {man['ood']['images']:,} images")
if not man["ood"]["images"]:
    print("  !! No out-of-domain set - the headline generalisation number will be missing.")

for split in C.SPLITS:
    n = sum(len(list((C.WORK_DIR / split / c).glob('*'))) for c in man["labels"])
    print(f"{split:<18}: {n:,}")

print("\nper-class (train/val/test):")
for label in man["labels"]:
    counts = [len(list((C.WORK_DIR / s / label).glob('*'))) for s in C.SPLITS]
    flag = "healthy" if C.is_healthy(label) else "UNHEALTHY"
    print(f"  {label:<48} {counts[0]:>5}/{counts[1]:>4}/{counts[2]:>4}  {flag}")

## 4. Train

Transfer learning on MobileNetV3-Small (ImageNet weights), two phases:

1. backbone frozen, train the new classifier head
2. unfreeze the top 30% of the backbone and fine-tune at a much lower learning rate

Class weights compensate for PlantVillage's imbalance. Augmentation is limited to
realistic field variation - no hue shifts or heavy colour jitter, because colour
*is* the signal for several diseases.

Add `--quick` for a short smoke run when you just want to check the plumbing.

In [ ]:
!python scripts/train.py

## 5. Evaluate - the honest part

Evaluated on two sets:

* **in-domain**: the held-out PlantVillage test split
* **out-of-domain**: PlantDoc field photographs, never trained on - this is the
  number that predicts webcam behaviour

In [ ]:
!python scripts/evaluate.py

In [ ]:
# Confusion matrices written by evaluate.py
from IPython.display import Image, display
import config as C

for stem in ("in_domain_test", "ood_plantdoc"):
    p = C.REPORTS_DIR / f"confusion_{stem}.png"
    if p.exists():
        print(stem)
        display(Image(filename=str(p)))
    else:
        print(f"{stem}: no confusion matrix (split not available)")

## 6. Export to TensorFlow Lite

Converts the trained Keras model and then **verifies the conversion**: it runs
both the Keras model and the TFLite model over the same test images and reports
top-1 agreement and the maximum probability drift. A conversion that changed
predictions is a failed conversion, whatever the file size says.

In [ ]:
!python scripts/export_tflite.py

## 7. Final report

Everything below is read from the evaluation JSON that was just written - copy this output back to me verbatim.

In [ ]:
import json
from pathlib import Path
import config as C

man = json.loads((C.DATASET_DIR / "manifest.json").read_text())
ev  = json.loads((C.REPORTS_DIR / "evaluation.json").read_text())
tc  = json.loads((C.MODELS_DIR / "training_config.json").read_text())
exp = json.loads((C.MODELS_DIR / "export" / "class_names.json").read_text()).get("export", {})

pct = lambda v: "n/a" if v is None else f"{v*100:.2f}%"
num = lambda v: "n/a" if v is None else f"{v:.4f}"

print("=" * 72)
print("AGRIVISION MODEL 2 - PLANT HEALTH CLASSIFIER - TRAINING REPORT")
print("=" * 72)

print("\nA. DATASET")
print(f"   train/val/test : PlantVillage (public domain), via TFDS 'plant_village'")
print(f"   out-of-domain  : PlantDoc (CC-BY-4.0) - evaluation only, never trained on")

print("\nB. IMAGES")
for split in C.SPLITS:
    n = sum(len(list((C.WORK_DIR / split / c).glob('*'))) for c in man["labels"])
    print(f"   {split:<6}: {n:,}")
print(f"   OOD   : {man['ood']['images']:,}")
print(f"   healthy {man['healthy_images']:,} / unhealthy {man['unhealthy_images']:,}")

print(f"\nC. CLASSES: {man['num_classes']}")
for label in man["labels"]:
    print(f"   {label}")

print("\nD. TRAINING METHOD")
print(f"   backbone   : {C.BACKBONE} (ImageNet), include_preprocessing=True")
print(f"   input      : {C.IMG_SIZE}x{C.IMG_SIZE}, raw 0-255 float")
print(f"   phase 1    : head only, {tc.get('epochs_head')} epochs @ lr {C.LR_HEAD}")
print(f"   phase 2    : fine-tune top {int((1-C.FINETUNE_AT)*100)}%, "
      f"{tc.get('epochs_finetune')} epochs @ lr {C.LR_FINETUNE}")
print(f"   best val acc during training: {pct(tc.get('best_val_accuracy'))}")

for key, title in (("in_domain_test_plantvillage", "E. IN-DOMAIN TEST (PlantVillage)"),
                   ("out_of_domain_plantdoc",      "E. OUT-OF-DOMAIN TEST (PlantDoc)  <-- HEADLINE")):
    r = ev.get(key)
    print(f"\n{title}")
    if not r:
        print("   NOT AVAILABLE")
        continue
    print(f"   images            : {r['images']:,}")
    print(f"   accuracy          : {pct(r['accuracy'])}")
    print(f"   precision (macro) : {num(r['precision_macro'])}")
    print(f"   recall    (macro) : {num(r['recall_macro'])}")
    print(f"   F1        (macro) : {num(r['f1_macro'])}")
    print(f"   F1     (weighted) : {num(r['f1_weighted'])}")
    h = r["health"]
    print(f"   -- HEALTHY vs UNHEALTHY (what the app shows) --")
    print(f"   health accuracy   : {pct(h['health_accuracy'])}")
    print(f"   precision         : {num(h['precision_unhealthy'])}  (positive = UNHEALTHY)")
    print(f"   recall            : {num(h['recall_unhealthy'])}")
    print(f"   F1                : {num(h['f1_unhealthy'])}")
    cm = h["confusion_matrix"]
    print(f"   confusion         :          pred UNHEALTHY   pred HEALTHY")
    print(f"     true UNHEALTHY            {cm[0][0]:>8}       {cm[0][1]:>8}")
    print(f"     true HEALTHY              {cm[1][0]:>8}       {cm[1][1]:>8}")
    print(f"   missed disease    : {pct(h['missed_disease_rate_unhealthy_called_healthy'])}")
    print(f"   false alarm       : {pct(h['false_alarm_rate_healthy_called_unhealthy'])}")

ind, ood = ev.get("in_domain_test_plantvillage"), ev.get("out_of_domain_plantdoc")
print("\nH. DOES IT GENERALISE?")
if ind and ood:
    gap = ind["accuracy"] - ood["accuracy"]
    print(f"   in-domain {pct(ind['accuracy'])} vs field {pct(ood['accuracy'])} -> gap {gap*100:.1f} points")
    if gap > 0.25:
        print("   VERDICT: NO. The model has substantially learned PlantVillage's")
        print("            capture conditions, not the disease itself. Treat the")
        print("            field number as the real one and do not ship this as a")
        print("            diagnostic tool.")
    elif ood["accuracy"] < 0.60:
        print("   VERDICT: WEAK. Field accuracy is too low to rely on.")
    else:
        print("   VERDICT: reasonable, but PlantDoc is still leaf-centric photography.")
else:
    print("   Cannot judge - the out-of-domain set is missing.")

print("\nI. EXPORT VERIFICATION (TFLite vs Keras)")
if exp:
    print(f"   {json.dumps(exp, indent=3)}")
else:
    print("   export metadata not found - did scripts/export_tflite.py run?")

print("\nPER-CLASS (out-of-domain where available, else in-domain)")
src = ood or ind
if src:
    print(f"   {'class':<46}{'prec':>7}{'rec':>7}{'f1':>7}{'n':>7}")
    for name, m in src["per_class"].items():
        if not isinstance(m, dict) or "precision" not in m:
            continue
        print(f"   {name:<46}{m['precision']:>7.2f}{m['recall']:>7.2f}"
              f"{m['f1-score']:>7.2f}{int(m['support']):>7}")
print("=" * 72)

## 8. Download the two files

In [ ]:
import shutil
from google.colab import files
import config as C

exp_dir = C.MODELS_DIR / "export"
for f in ("plant_health_classifier.tflite", "class_names.json"):
    p = exp_dir / f
    assert p.exists(), f"{p} missing - export step did not complete"
    print(f"{f:<34}{p.stat().st_size/1e6:>8.2f} MB")

shutil.make_archive("plant_health_model", "zip", exp_dir)
files.download("plant_health_model.zip")

## 9. Install into the AGRIVISION app

Unzip and copy **both** files into the repo's `model/` folder:

```
cv-prototype/model/plant_health_classifier.tflite
cv-prototype/model/class_names.json
```

Both are required. The browser reads `class_names.json` for the label order, the
input range and the abstention thresholds - it never hard-codes them - so a
mismatched pair is worse than no model at all.

Then reload the app. `PlantAnalyzer` fetches the metadata on startup: if it 404s
the health card says **HEALTH MODEL NOT AVAILABLE** and detection carries on
unaffected; when both files are present, every plant Model 1 confirms is cropped
and classified, and the card shows HEALTHY / UNHEALTHY with its confidence.
